In [105]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool, cv
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv("./train.csv")
test = pd.read_csv("./test.csv")

# Create a combined dataset for consistent feature engineering
combined = pd.concat([df, test], sort=False)
combined["IsTrain"] = combined["Survived"].notna()

# Advanced Feature Engineering
# 1. Title extraction from Name
combined["Title"] = combined["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
combined["Title"] = combined["Title"].replace(
    [
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    ],
    "Rare",
)
combined["Title"] = combined["Title"].replace(["Mlle", "Ms"], "Miss")
combined["Title"] = combined["Title"].replace("Mme", "Mrs")

# 2. Family size and categories
combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1
combined["IsAlone"] = (combined["FamilySize"] == 1).astype(int)
combined["FamilySizeCategory"] = pd.cut(
    combined["FamilySize"],
    bins=[0, 1, 4, 7, 20],
    labels=["Single", "Small", "Medium", "Large"],
)

# 3. Cabin features (if available)
combined["HasCabin"] = combined["Cabin"].notna().astype(int)
combined["CabinLetter"] = combined["Cabin"].str[0].fillna("U")
combined["CabinLetter"] = combined["CabinLetter"].replace(["T", "U"], "U")

# 4. Age grouping and interactions
combined["AgeGroup"] = pd.cut(
    combined["Age"],
    bins=[0, 12, 18, 35, 50, 80],
    labels=["Child", "Teen", "Adult", "Middle", "Senior"],
)

# 5. Fare grouping (for test data)
combined["FareGroup"] = pd.qcut(
    combined["Fare"].fillna(combined["Fare"].median()),
    q=4,
    labels=["Low", "Medium", "High", "Very_High"],
)

# 6. Ticket features
combined["TicketPrefix"] = (
    combined["Ticket"].str.extract("([A-Za-z]+)\.?", expand=False).fillna("NUM")
)
combined["TicketLength"] = combined["Ticket"].str.len()

# 7. Interaction features
combined["Age*Pclass"] = combined["Age"] * combined["Pclass"]
combined["FarePerPerson"] = combined["Fare"] / combined["FamilySize"]
combined["Sex_Pclass"] = combined["Sex"] + "_" + combined["Pclass"].astype(str)

# Fill missing values
combined["Age"] = combined.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)
combined["Age"] = combined["Age"].fillna(combined["Age"].median())

combined["Fare"] = combined.groupby(["Pclass", "Embarked"])["Fare"].transform(
    lambda x: x.fillna(x.median())
)
combined["Fare"] = combined["Fare"].fillna(combined["Fare"].median())

combined["Embarked"] = combined["Embarked"].fillna(combined["Embarked"].mode()[0])

# Drop unnecessary columns
drop_cols = ["Name", "Ticket", "Cabin", "PassengerId"]
if "Survived" in combined.columns:
    drop_cols.append("Survived")

# Split back
df_train = combined[combined["IsTrain"]].copy()
df_test = combined[~combined["IsTrain"]].copy()

# Prepare features
X = df_train.drop(columns=["Survived", "IsTrain"] + drop_cols)
y = df_train["Survived"]

# Categorical features
cat_features = [
    "Sex",
    "Embarked",
    "Title",
    "FamilySizeCategory",
    "CabinLetter",
    "AgeGroup",
    "FareGroup",
    "TicketPrefix",
    "Sex_Pclass",
]

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score


def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "subsample": trial.suggest_float("subsample", 0.5, 1),
        "loss_function": "Logloss",
        "eval_metric": "Accuracy",
        "random_seed": 42,
        "verbose": 0,
    }

    # Cross-validation
    cv_scores = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=0,
        )

        preds = model.predict(X_val)
        cv_scores.append(accuracy_score(y_val, preds))

    return np.mean(cv_scores)


# Run optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=3600)

print("Best parameters:", study.best_params)
print("Best CV score:", study.best_value)

# Train with best parameters
best_model = CatBoostClassifier(**study.best_params)
best_model.fit(X, y, cat_features=cat_features, verbose=100)

: 

: 

: 

: 

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# Prepare data (ensure numerical encoding)
X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=True)
X_test_encoded = pd.get_dummies(
    df_test.drop(columns=["IsTrain"] + drop_cols), columns=cat_features, drop_first=True
)

# Align columns
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# Create individual models
model_catboost = CatBoostClassifier(**study.best_params, verbose=0, random_seed=42)
model_xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
)
model_lgbm = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1
)

# Voting Classifier
voting_clf = VotingClassifier(
    estimators=[("catboost", model_catboost), ("xgb", model_xgb), ("lgbm", model_lgbm)],
    voting="soft",
)

voting_clf.fit(X_encoded, y)

# Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=[("catboost", model_catboost), ("xgb", model_xgb), ("lgbm", model_lgbm)],
    final_estimator=LogisticRegression(),
    cv=5,
)

stacking_clf.fit(X_encoded, y)

# Make predictions
preds_voting = voting_clf.predict(X_test_encoded)
preds_stacking = stacking_clf.predict(X_test_encoded)

: 

: 

In [ ]:
# 1. Try target encoding for high-cardinality features
from category_encoders import TargetEncoder

te = TargetEncoder()
X["TicketPrefix_encoded"] = te.fit_transform(X["TicketPrefix"], y)
X_test["TicketPrefix_encoded"] = te.transform(df_test["TicketPrefix"])

# 2. Create more interaction features
X["Age_Pclass_Sex"] = X["Age"] * X["Pclass"] * (X["Sex"] == "female").astype(int)
X_test["Age_Pclass_Sex"] = (
    X_test["Age"] * X_test["Pclass"] * (X_test["Sex"] == "female").astype(int)
)

# 3. Use probability calibration
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(best_model, method="platt", cv=5)
calibrated_model.fit(X_encoded, y)

# 4. Post-processing - adjust threshold
from sklearn.metrics import f1_score

# Find optimal threshold
thresholds = np.arange(0.3, 0.7, 0.01)
best_threshold = 0.5
best_f1 = 0

probs = best_model.predict_proba(X_encoded)[:, 1]
for threshold in thresholds:
    preds_threshold = (probs > threshold).astype(int)
    f1 = f1_score(y, preds_threshold)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Optimal threshold: {best_threshold}")

# Use optimal threshold for final predictions
probs_test = best_model.predict_proba(X_test_encoded)[:, 1]
preds_optimized = (probs_test > best_threshold).astype(int)

: 

: 

In [ ]:
# Final submission with best model
submission = pd.DataFrame(
    {
        "PassengerId": df_test["PassengerId"],
        "Survived": preds_optimized,  # Use predictions from best model
    }
)
submission.to_csv("submission_improved.csv", index=False)

# If using ensemble
submission_ensemble = pd.DataFrame(
    {
        "PassengerId": df_test["PassengerId"],
        "Survived": preds_voting,  # or preds_stacking
    }
)
submission_ensemble.to_csv("submission_ensemble.csv", index=False)

: 

: 